# Purchase cohort retention

демо-реконструкция подхода на синтетических данных

Use RETAIL_DATASET for the public UCI source; otherwise the run explicitly uses fictional data. See DATA_SOURCES.md.


In [1]:
import os, sys
from pathlib import Path
root = Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parent
sys.path.insert(0, str(root))
from src.data import load_transactions
transactions, source_label = load_transactions(os.getenv('RETAIL_DATASET'))
print(source_label)
print('Rows:', len(transactions))

Synthetic fallback fixture, not open-data results
Rows: 134


In [2]:
import pandas as pd
activity=transactions[['customer','date']].copy()
activity['month']=activity.date.dt.to_period('M')
activity['cohort']=activity.groupby('customer')['month'].transform('min')
activity['age']=(activity.month.dt.year-activity.cohort.dt.year)*12+activity.month.dt.month-activity.cohort.dt.month
counts=activity.groupby(['cohort','age']).customer.nunique().unstack()
last_month=activity.month.max()
# Unobserved future cohort ages remain NaN, never zero retention.
for cohort in counts.index:
    observed_age=(last_month.year-cohort.year)*12+last_month.month-cohort.month
    for age in counts.columns:
        if age<=observed_age and pd.isna(counts.loc[cohort,age]): counts.loc[cohort,age]=0
retention=counts.div(counts[0],axis=0)
assert (retention[0]==1).all()
print(retention.to_string())

age        0         1         2         3    4         5
cohort                                                   
2026-01  1.0  0.636364  0.681818  0.681818  1.0  0.636364
2026-02  1.0  1.000000  1.000000  0.000000  1.0       NaN
